## Lecture 3: Optimization and Numba

### ****Milestone 1:** Function-level profiling with `cProfile`**

**Use cProfile on both versions:**

In [1]:
# From slide 27 (my function are named differently including max_iter)

import cProfile
import pstats
from mandelbrot import compute_mandelbrot as mandelbrot_naive
from mandelbrot import compute_mandelbrot_numpy as mandelbrot_numpy

cProfile.run('mandelbrot_naive(-2, 1, -1.5, 1.5, 512, 512, 100)',
             'naive_profile.prof')

cProfile.run('mandelbrot_numpy(-2, 1, -1.5, 1.5, 512, 512, 100)',
             'numpy_profile.prof')

for name in ('naive_profile.prof', 'numpy_profile.prof'):
    stats = pstats.Stats(name)
    stats.sort_stats('cumulative')
    stats.print_stats(10)

Mon Mar  2 21:15:47 2026    naive_profile.prof

         6223626 function calls in 1.951 seconds

   Ordered by: cumulative time
   List reduced from 20 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    1.951    1.951 {built-in method builtins.exec}
        1    0.000    0.000    1.951    1.951 <string>:1(<module>)
        1    0.435    0.435    1.950    1.950 /ncs/mandelbrot.py:39(compute_mandelbrot)
   262144    1.197    0.000    1.501    0.000 /ncs/mandelbrot.py:29(mandelbrot_point)
  5698796    0.304    0.000    0.304    0.000 {built-in method builtins.abs}
   262656    0.014    0.000    0.014    0.000 {method 'append' of 'list' objects}
        2    0.000    0.000    0.000    0.000 /opt/conda/envs/nsc/lib/python3.11/site-packages/numpy/_core/function_base.py:26(linspace)
        2    0.000    0.000    0.000    0.000 /opt/conda/envs/nsc/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3523(nd

**Questions to answer: (slide 27)**

- **Which function takes most total time?**

*Naïve:* `mandelbrot_point`, which is to be expected.

*Numpy:* `/ncs/mandelbrot.py:61(compute_mandelbrot_numpy)` which is the function itself.

- **Are there functions called surprisingly many times?**

Yes. `mandelbrot_point`, `append` are called tens of thousands of times, with `abs` being called 10x more often. That is a lot of function calls!

- **How does the NumPy profile compare to naive?**

From the output, it appears that numpy is able to compute most of the work inside of itself, without python being able "to see" it i.e., number of function calls. I am guessing that this is also due to the binary mask.

****Done?** Add the cProfile output for both versions and your answers to the questions above to your mini-report → commit**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit 3701704954893c917622f4ffc45c6e5343908271 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 21:17:26 2026 +0000

    l03: milestone 1, half done
```

**Questions to answer: (slide 28)**

**1. Which function takes most total time?**

Please see the previous answers above.

**2. Are there functions called surprisingly many times?**

Please see the previous answers above.

**3. How does NumPy profile compare to naive?**

Please see the previous answers above.

**4. Where does NumPy spend its time?**

`/ncs/mandelbrot.py` which is the function itself.

**Document your findings! Add a summary table to your mini-report.**

### ****Milestone 2:** Line-level profiling with `line_profiler`**

**Add `@profile` decorator and run line profiler:**

In [2]:
import numpy as np
from memory_profiler import profile

# from slide 29


@profile  # Add this decorator
def mandelbrot_naive_profile(xmin, xmax, ymin, ymax, width, height, max_iter=100):
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    result = np.zeros((height, width), dtype=int)
    for i in range(height):
        for j in range(width):
            c = x[j] + 1j * y[i]
            z = 0
            for n in range(max_iter):
                if abs(z) > 2:
                    result[i, j] = n
                    break
                z = z*z + c
        else:
            result[i, j] = max_iter

    return result

```sh
kernprof -l -v mandelbrot_script.py # or use Spyder: Run -> Profile (F10)
```

Before we can run kernprof, we need to install it from the line_profiler package.

In [3]:
!mamba clean --all --yes

Collect information..
Cleaning index cache..
Cleaning lock files..
Cleaning tarballs..
Cleaning packages..


In [4]:
!mamba install --name nsc line_profiler -y

[+] 0.0s
[+] 0.1s
conda-forge/linux-64 ━━╸━━━━━━━━━━━━━━━╸━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.0s
conda-forge/noarch   ━━━━━━━━━╸━━━━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.0s[+] 0.2s
conda-forge/linux-64 ━━━━━━━━━━━━━━━━━━━━━━━ 401.4kB /  51.4MB @   2.6MB/s  0.1s
conda-forge/noarch   ━━━━━━━━━━━╸━━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.1s[+] 0.3s
conda-forge/linux-64 ━━━━━━━━━━━━━━━━━━━━━━━   1.6MB /  51.4MB @   6.2MB/s  0.2s
conda-forge/noarch   ━━━━━━━━━━━━━━━━━━━━━━━ 210.6kB /  24.9MB @ 813.9kB/s  0.2s[+] 0.4s
conda-forge/linux-64 ━━━━━━━━━━━━━━━━━━━━━━━   2.2MB /  51.4MB @   6.3MB/s  0.3s
conda-forge/noarch   ━━━━━━━━━━━━━━━━━━━━━━━ 303.1kB /  24.9MB @ 866.5kB/s  0.3s[+] 0.5s
conda-forge/linux-64 ╸━━━━━━━━━━━━━━━━━━━━━━   3.8MB /  51.4MB @   8.3MB/s  0.4s
conda-forge/noarch   ━━━━━━━━━━━━━━━━━━━━━━━ 772.5kB /  24.9MB @   1.7MB/s  0.4s[+] 0.6s
conda-forge/linux-64 ━╸━━━━━━━━━━━━━━━━━━━━━   4.9MB /  51.4MB @   8.9MB/s  0.5s
conda-forge/noarch   ━━━━━━━━━━━━━━━━━━━━━━━   1.1M

In [5]:
# Profile the function. Do not profile the profiler (See table on slide 30)
from mandelbrot import compute_mandelbrot as mandelbrot_naive
from mandelbrot import compute_mandelbrot_numpy as mandelbrot_numpy

In [6]:
%load_ext line_profiler

In [7]:
%lprun -f mandelbrot_naive mandelbrot_naive(-2, 1, -1.5, 1.5, 512, 512, 100)

Timer unit: 1e-09 s

Total time: 2.59478 s
File: /ncs/mandelbrot.py
Function: compute_mandelbrot at line 39

Line #      Hits         Time  Per Hit   % Time  Line Contents
    39                                           def compute_mandelbrot(
    40                                               x_min: float,
    41                                               x_max: float,
    42                                               y_min: float,
    43                                               y_max: float,
    44                                               width: int,
    45                                               height: int,
    46                                               max_iter: int
    47                                           ) -> list[list[int]]:
    48         1     179637.0 179637.0      0.0      x = np.linspace(x_min, x_max, width)
    49         1      52749.0  52749.0      0.0      y = np.linspace(y_min, y_max, height)
    50                               

Load the line profiler into ipython, available via the magic command lprun.

We can see that appending and computing the mandelbrot point is very expensive in terms of time, taking $99\%$ of the total running time.

****Done?** Commit → continue to profiling report**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit cf632815d5ff453831b97b0f40034955dc9a3ef0 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 21:32:01 2026 +0000

    l03: milestone 2, half done
```

****Rule:** Use `line_profiler` to find *which line* is the bottleneck. Once you know, **remove** `@profile` and benchmark with plain `perf_counter`**

**Support your answers with numbers from your profiling output — copy-pasting terminal output directly into your mini-report is fine.**

**Add a profiling section to your MP1 mini-report. In your own words, answer:**

**1. **cProfile on naive vs NumPy:** How many functions appear in each profile? What does this difference tell you about where the work actually happens?**

In [8]:
# Mostly from slide 27

from mandelbrot import compute_mandelbrot as mandelbrot_naive
from mandelbrot import compute_mandelbrot_numpy as mandelbrot_numpy

cProfile.run('mandelbrot_naive(-2, 1, -1.5, 1.5, 512, 512, 100)',
             'naive_profile.prof')

cProfile.run('mandelbrot_numpy(-2, 1, -1.5, 1.5, 512, 512, 100)',
             'numpy_profile.prof')

for name in ('naive_profile.prof', 'numpy_profile.prof'):
    stats = pstats.Stats(name)
    print(f"{name}: {len(stats.stats)}")

naive_profile.prof: 20
numpy_profile.prof: 42


It appears that numpy has more than twice the number of functions involved. I assume that this is due to numpy being more fragmented, where functions are only good at one thing.

**2. **`line_profiler` on naive:** Which lines dominate runtime? What fraction of total time is spent in the inner loop?**

In [9]:
%lprun -f mandelbrot_naive mandelbrot_naive(-2, 1, -1.5, 1.5, 512, 512, 100) # same as before

Timer unit: 1e-09 s

Total time: 2.32989 s
File: /ncs/mandelbrot.py
Function: compute_mandelbrot at line 39

Line #      Hits         Time  Per Hit   % Time  Line Contents
    39                                           def compute_mandelbrot(
    40                                               x_min: float,
    41                                               x_max: float,
    42                                               y_min: float,
    43                                               y_max: float,
    44                                               width: int,
    45                                               height: int,
    46                                               max_iter: int
    47                                           ) -> list[list[int]]:
    48         1     149611.0 149611.0      0.0      x = np.linspace(x_min, x_max, width)
    49         1      51707.0  51707.0      0.0      y = np.linspace(y_min, y_max, height)
    50                               

We can see that line 55 is *the* dominant runtime.

The inner loop is line 55-56, spending $100\%$.

**3. Based on your profiling results: why is NumPy faster than naive Python?**

Numpy moves the inner loop into compiled C code (broadcasting), which is much faster than pure python.

**4. What would you need to change to make the naive version faster? (hint: what does line profiler tell you about the inner loop?)**

`mandelbrot_point` and python `.append()` should be reworked. For example, preallocating the grid/image and inlining the point computation, perhaps even compiling it.

****Done?** Commit → move on to Milestone 3 (Numba)**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit b63cc9b52486ab03b0c56e946d2b5e27b165f847 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 21:38:44 2026 +0000

    l03: milestone 2 done
```

### ****Milestone 3:** Implement Numba `@njit` and benchmark**

In [10]:
from numba import njit


# From slide 36. Approach A (where I have filled in the gaps in the hybrid)
@njit
def mandelbrot_point_numba(c, max_iter=100):
    z = 0j
    for n in range(max_iter):
        if z.real*z.real+z.imag*z.imag > 4.0:
            return n
        z = z*z + c
    return max_iter


def mandelbrot_hybrid(xmin, xmax, ymin, ymax, width, height, max_iter=100):
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    result = np.zeros((height, width), dtype=np.int32)

    # outer loops still in Python
    for i in range(height):
        for j in range(width):
            c = x[j] + 1j * y[i]
            result[i, j] = mandelbrot_point_numba(c, max_iter)

In [11]:
# from slide 37. Approach B
from numba import njit
import numpy as np


@njit                       # <-- entire function compiled !
def mandelbrot_naive_numba(xmin, xmax, ymin, ymax, width, height, max_iter=100):
    """Fully JIT - compiled Mandelbrot --- structure identical to naive."""
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    result = np.zeros((height, width), dtype=np.int32)

    for i in range(height):             # compiled loop
        for j in range(width):          # compiled loop
            c = x[j] + 1j * y[i]
            z = 0j                      # complex literal: type inference works!
            n = 0
            while n < max_iter and (z.real*z.real + z.imag*z.imag) <= 4.0:
                z = z*z + c
                n += 1
        result[i, j] = n
    return result


# Warm - up (triggers compilation --- don ’t time this!)
_ = mandelbrot_naive_numba(-2, 1, -1.5, 1.5, 64, 64)

****Your Task:** Implement both approaches and measure the difference.**

In [12]:
# From slide 38
import time
import statistics


def bench(fn, *args, runs=5):
    fn(*args)  # extra warm-up
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


# Warm up (triggers JIT compilation -- exclude from timing)
_ = mandelbrot_hybrid(-2, 1, -1.5, 1.5, 64, 64)
_ = mandelbrot_naive_numba(-2, 1, -1.5, 1.5, 64, 64)

t_hybrid = bench(mandelbrot_hybrid, -2, 1, -1.5, 1.5, 1024, 1024)
t_full = bench(mandelbrot_naive_numba, -2, 1, -1.5, 1.5, 1024, 1024)

print(f"Hybrid:         {t_hybrid:.3f}s")
print(f"Fully compiled: {t_full:.3f}s")
print(f"Ratio:          {t_hybrid/t_full:.1f}x")

Hybrid:         1.509s
Fully compiled: 0.045s
Ratio:          33.3x


Since we are provided with the code, I assume that this is the code I *must* use.

****Done?** Commit → log in Performance Tracker**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit 97144fb7739f7214faf838257d394995d9b084f4 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 21:42:59 2026 +0000

    l03: milestone 3 almost done
```

In [13]:
# From slide 39, but with max iter included
import time
import statistics


def bench(fn, *args, runs=5):
    fn(*args)  # warm-up
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


width, height = 1024, 1024
args = (-2, 1, -1.5, 1.5, width, height, 100) # include max iter

t_naive = bench(mandelbrot_naive,           *args)
t_numpy = bench(mandelbrot_numpy,           *args)
t_numba = bench(mandelbrot_naive_numba,     *args)

print(f"Naive: {t_naive:.3f}s")
print(f"NumPy: {t_numpy:.3f}s ({t_naive/t_numpy:.1f}x)")
print(f"Numba: {t_numba:.3f}s ({t_naive/t_numba:.1f}x)")

Naive: 4.113s
NumPy: 0.573s (7.2x)
Numba: 0.048s (86.6x)


****Done?** Add timings and speedup table (naive / NumPy / Numba) to your mini-report → log in
Performance Tracker → commit**

```sh
┌──(mikkel@pella)-[~/Sync/AAU/Semester 8/github/nsc-mikkel]
└─$ git log -n 1
commit eec7018d9c7270f98d237026c39807f494a13604 (HEAD -> main)
Author: mikkel-coder <mikkel.ks.sorensen@gmail.com>
Date:   Mon Mar 2 21:47:43 2026 +0000

    l03: milestone 3 done
```

### ****Milestone 4:** Data type optimization (float32 vs float64)**

**Modify your Numba version to accept a dtype parameter:**

**Compare accuracy and generate side-by-side images:**

**Run your experiments and observe:**

- ****Speed:** Does float32 actually run faster than float64 on your hardware? By how much? Is float16 faster or slower — and why might that surprise you?**

- ****Visual quality:** Zoom in on a detailed region of the Mandelbrot set. Can you see artefacts with float16? What about float32?**

- ****Recommendation:** Based on what you observe, which precision would you choose for production use, and why?**

**Hint: The answer is not always obvious — modern CPUs handle different precisions differently, and the memory bandwidth argument does not always dominate.**

**Add a data type section to your MP1 mini-report. Include:**

- **Your measured runtimes for float16, float32, float64**

- **Side-by-side images (or a zoomed crop showing where artefacts appear)**

- **A brief recommendation: which precision would you use and why?**

**No specific format required — a few sentences and your images is enough. The goal is to show you understand the precision/performance trade-off.**

****Done?** Add and commit work and results.**

### ****Milestone 5 (extension):** Parallel Numba with `prange`**

#### ****Experiment 1:** `@jit` vs. `@njit`**

- **Replace `@njit` with `@jit`**

- **Measure performance difference**

- **Usually `@njit` is faster**

#### ****Experiment 2:** Parallel Numba (advanced)**

- **Try `parallel=True` and `prange`**

- **May give 2-4× speedup on multi-core**

#### ****Experiment 3:** Scaling with grid size**

- **Test $512^2$, $1024^2$, $4096^2$ (or larger)**

- **Plot grid size vs. time for all implementations**

- **Does speedup scale with problem size?**